# GuacaMol Dataset Tutorial

[GuacaMol](https://github.com/BenevolentAI/guacamol) (Brown et al., 2019) is a benchmark dataset
derived from ChEMBL containing ~1.6 million drug-like SMILES strings. Each molecule has 10 RDKit
physicochemical properties: `BertzCT`, `MolLogP`, `MolWt`, `TPSA`, `NumHAcceptors`, `NumHDonors`,
`NumRotatableBonds`, `NumAliphaticRings`, `NumAromaticRings`, and `QED`.

This tutorial demonstrates the `GuacaMol` dataset class from `alf_tools`:
1. Downloading and inspecting the raw SMILES files
2. Loading the dataset and building a pandas DataFrame
3. Visualising property distributions and correlations
4. Querying properties for arbitrary molecules
5. Case study: aspirin's physicochemical profile

We use `max_molecules=10_000` throughout so the notebook runs in under a minute on CPU.

## Section 0 — Environment Setup

Run `uv sync` from the `tutorials/` directory to install all dependencies, then select the `.venv`
kernel when prompted.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    ["uv", "pip", "install", "--python", sys.executable, "-e", ".."],
)
print("✓ Environment ready")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display as ipy_display
from rdkit import Chem
from rdkit.Chem.Draw import MolToImage

from alf_core import Candidate
from alf_core.dataclasses.candidate import Modality
from alf_tools.datasets.guacamol import (
    ALL_PROPERTIES,
    GUACAMOL_FILES,
    GuacaMol,
    GuacaMolConfig,
    _compute_properties,
    download_guacamol,
)

DATA_DIR = Path.home() / ".cache" / "alf"
PROPERTY_COLS = sorted(ALL_PROPERTIES)

print("✓ Imports OK")
print(f"Properties ({len(PROPERTY_COLS)}): {PROPERTY_COLS}")

## Section 1 — Download

`download_guacamol` streams all four GuacaMol SMILES files from Figshare and caches them locally.
With `max_lines=10_000`, only the first 10 000 lines of each file are written to disk; subsequent
runs detect the existing files and skip the download entirely.

In [ ]:
download_guacamol(data_dir=DATA_DIR, max_lines=10_000)

print("Downloaded files:")
for split, info in GUACAMOL_FILES.items():
    filepath = DATA_DIR / info["name"]
    size_kb = filepath.stat().st_size / 1024
    print(f"  {split:5s}  {info['name']}  ({size_kb:.1f} KB)")

In [ ]:
all_smiles_path = DATA_DIR / GUACAMOL_FILES["ALL"]["name"]
with open(all_smiles_path, encoding="utf-8") as f:
    sample = [next(f).strip() for _ in range(5)]

print("First 5 SMILES from the corpus:")
for i, smi in enumerate(sample, 1):
    print(f"  {i}. {smi}")